## Figure 2. Feature selection and coefficients underlying the NMWI. 
Feature selection and coefficients underlying the NMWI. **(a)** LASSO-penalized logistic regression coefficients: 14 taxa with positive coefficients, 10 with negative coefficients, and 160 with coefficients shrunk to zero. Bubble size indicates selection frequency across LOSO CV, and color indicates coefficient magnitude. **(b)** Heatmap of coefficients across LOSO CV models, in which each column represents a model trained with one study held out; and each row represents a taxon in the final model. Coefficients are represented using the same color scale as in panel **(a)**.


In [3]:
from pathlib import Path
import os, runpy, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns

from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API")

project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "NMWI")
os.chdir(project_root)

In [4]:
%matplotlib inline
# METADATA LOADING
vars_dict = runpy.run_path("Codes/Auxiliary_files/Generate_data_index.py")
globals().update(vars_dict)
X_new = X.copy()

Found 27 genus tables in index
Number of samples: 1654, and number of taxa: 184
Number of unique studies: 27

Sample ID per Category:
Category  n_Sample ID
    case         1065
 control          589


- X: DataFrame (rows=samples, cols=taxonomy), index includes bioproject/sample_id RELATIVE ABUNDANCE
- y: Series/array-like of True/False (same index as X), True (1) =control, False=case

### **Hyperparameters search** 
Regularization strength for the L1 penalized logistic regression model was optimized using leave-one-study-out cross validation and accuracy. An initial broad search (C = [0–100]) across values of the inverse regularization parameter, C, was used to identify the region associated with the highest cross validated performance. Posteriorly, fine grid search within the selected range, evaluating C from 0.0025 to 4 in increments of 0.0025. For each value of C, the model was trained repeatedly while holding out one study at a time, and the mean accuracy across held out studies was calculated. The value of C yielding the highest mean LOSO accuracy was selected for the final NMWI model. 

In [ ]:
# --------------------------------------------------
# LOSO GRID SEARCH FOR BEST C 
# (Takes a while to run)
# --------------------------------------------------

# Base parameters shared by all models
model_params = dict(
    random_state=42,
    penalty="l1",
    solver="liblinear",
    class_weight="balanced",
    max_iter=2000
)

# Grid search for parameters
C_grid = np.arange(0.0025, 4.0025, 0.0025)

# Groups for LOSO
bio_ids = np.array(X_new.index.get_level_values(0))
loso = LeaveOneGroupOut()

X_arr = X_new
y_arr = y.values.ravel()

loso_results = []
loso_scores = np.full(len(y), np.nan)

for C_val in C_grid:
    fold_bas = []

    for train_idx, test_idx in loso.split(X_arr, y_arr, groups=bio_ids):
        X_tr = X_arr.iloc[train_idx]
        X_te = X_arr.iloc[test_idx]
        y_tr = y_arr[train_idx]
        y_te = y_arr[test_idx]

        model = LogisticRegression(**model_params, C=C_val)
        model.fit(X_tr, y_tr)

        loso_scores[test_idx] = model.decision_function(X_te)

        y_hat = model.predict(X_te)
        ba = balanced_accuracy_score(y_te, y_hat)
        fold_bas.append(ba)

    mean_loso_ba = np.mean(fold_bas)

    loso_results.append({
        "C": C_val,
        "mean_LOSO_BA": mean_loso_ba
    })

loso_results = pd.DataFrame(loso_results)
loso_grid_df = pd.DataFrame(loso_results.sort_values("mean_LOSO_BA", ascending=False))

# Best C according to LOSO
C = loso_grid_df.iloc[0]["C"]
best_loso_score = loso_grid_df.iloc[0]["mean_LOSO_BA"]
plt.plot(loso_results['C'],loso_results['mean_LOSO_BA'])

In [ ]:
print(f'Best regularization parameters: {C}')

---
### **NMWI - L1-regularized logistic regression model**

The final NMWI model fits the L1-regularized logistic regression to the complete development dataset using the selected regularization parameter (`C = 0.9375`) and extracts the model coefficients and intercept used to calculate the NMWI score. The script is executed below to make the fitted model and derived coefficient objects available in this notebook.

The implementation is stored in:

```bash
Codes/Auxiliary_files/NMWI.py
```

In [ ]:
%run -i Codes/Auxiliary_files/NMWI.py # Run model

In [ ]:
# NMWI coefficients and intercept (Table S7):
print(final_coef)
print(f"\nRegularized intercept: {NMWI.intercept_[0]:.4f}")

---
**Note:** 
Nested LOSO cross-validation is computationally intensive and was run separately using `Codes/Auxiliary_files/nested_LOSO.py`. The precomputed fold-level results are loaded below for downstream analysis and visualization. To reproduce the analysis:

```bash
python Codes/Auxiliary_files/nested_LOSO.py
```
---

In [ ]:
# Load precomputed LOSO coefficients
coef_df = pd.read_csv("Data/Output/nested_loso_coefficients.tsv", sep="\t", index_col="BioProject")

In [ ]:
plot_freq = sorted_coefficients[["Coefficient"]].rename(columns={"Coefficient": "coef"}).copy()
plot_freq["sel_freq"] = (coef_df != 0).mean().reindex(plot_freq.index).fillna(0)
plot_freq["order"] = np.arange(1, len(plot_freq) + 1)
plot_freq = plot_freq[plot_freq["coef"] != 0]

df_pos = plot_freq[plot_freq["coef"] > 0]
df_neg = plot_freq[plot_freq["coef"] < 0]

# Shared scale and colormap
y_max = plot_freq["coef"].abs().max() + 1
norm = TwoSlopeNorm(vmin=-y_max, vcenter=0, vmax=y_max)

cmap_custom = LinearSegmentedColormap.from_list(
    "coef_cmap",
    [(0.00, "#8a1a01"), (0.10, "#b52302"), (0.20, "#f5350a"),
     (0.30, "#fa6328"), (0.35, "#fa9128"), (0.40, "#fab028"),
     (0.45, "#f7bc4d"), (0.47, "#ffdb70"), (0.50, "#fff2cc"),
     (0.60, "#22A884"), (0.70, "#2A788E"), (0.80, "#414487"),
     (1.00, "#440154")],
    N=256,
)

# Coefficient plots
fig, axes = plt.subplots(1, 2, figsize=(8, 6), sharey=True)

for ax, data, title in zip(axes, [df_pos, df_neg], ["Positive", "Negative"]):
    sc = ax.scatter(data["order"], data["coef"], s=40 + 450*data["sel_freq"],
                    c=data["coef"], cmap=cmap_custom, norm=norm,
                    edgecolor="black", linewidth=0.4, alpha=0.9)
    ax.axhline(0, color="gray", linestyle="--")
    ax.set(xlabel="Feature order", title=title, ylim=(-y_max, y_max))

axes[0].set_ylabel("Coefficient")

cbar = fig.colorbar(sc, ax=axes, pad=0.03)
cbar.set_label("Logistic regression coefficient")

size_levels = [0.50, 0.75, 1.00]
handles = [plt.scatter([], [], s=40 + 450*s, facecolor="lightgray",
                       edgecolor="black", linewidth=0.4) for s in size_levels]

fig.legend(handles, [f"{s:.0%}" for s in size_levels],
           title="Selection frequency", loc="upper right",
           bbox_to_anchor=(1.15, 0.95))

plt.show()

# LOSO coefficient heatmap
proj_to_citation = dict(zip(df["BioProject Number"], df["Citation"]))
feature_order = [*df_pos.index, *df_neg.index]
group_order = coef_df[feature_order].abs().sum(axis=1).sort_values(ascending=False).index
heat = coef_df.loc[group_order, feature_order].T.rename(columns=proj_to_citation)

fig_w = max(12, 0.6 * heat.shape[1])
fig_h = max(8, 0.28 * heat.shape[0])

plt.figure(figsize=(fig_w, fig_h))
sns.heatmap(heat, cmap=cmap_custom, norm=norm, linewidths=0.05,
            linecolor="lightgray",
            cbar_kws={"label": "Logistic regression coefficient"})

plt.xlabel("Held-out LOSO group (BioProject)")
plt.ylabel("Top features")
plt.title("LOSO Coefficient Heatmap")
plt.tight_layout()
plt.show()

---